# GitHub Events ETL Pipeline
Extract → Transform → Load from GitHub REST API into PostgreSQL using `dlt`.

---

## 1. Imports

In [1]:
# %pip install requests dlt[postgres]
import requests
import time
import os

## 2. Extract — Fetch GitHub Events

In [2]:
REPO_URL = "https://api.github.com/repos/DataTalksClub/data-engineering-zoomcamp/events"

response = requests.get(REPO_URL)
data = response.json()

print(f"Fetched {len(data)} events")
data[0]  # Preview first record

Fetched 30 events


{'id': '15784552743',
 'type': 'WatchEvent',
 'actor': {'id': 109175663,
  'login': 'nisenogueira',
  'display_login': 'nisenogueira',
  'gravatar_id': '',
  'url': 'https://api.github.com/users/nisenogueira',
  'avatar_url': 'https://avatars.githubusercontent.com/u/109175663?'},
 'repo': {'id': 419661684,
  'name': 'DataTalksClub/data-engineering-zoomcamp',
  'url': 'https://api.github.com/repos/DataTalksClub/data-engineering-zoomcamp'},
 'payload': {'action': 'started'},
 'public': True,
 'created_at': '2026-09-25T21:20:20Z',
 'org': {'id': 72699292,
  'login': 'DataTalksClub',
  'gravatar_id': '',
  'url': 'https://api.github.com/orgs/DataTalksClub',
  'avatar_url': 'https://avatars.githubusercontent.com/u/72699292?'}}

## 3. Rate Limit
> Unauthenticated: 60 req/hour. With token: 5,000 req/hour.

In [3]:
remaining = requests.get("https://api.github.com/rate_limit").json()['rate']['remaining']
print(f"Remaining calls: {remaining}")

if remaining == 0:
    time.sleep(60)  # Wait for quota to reset

Remaining calls: 59


## 4. Authentication
> Store your token in an environment variable — never hardcode it.

In [4]:
API_TOKEN = os.getenv("GITHUB_TOKEN", "")  # Set via env var
headers = {"Authorization": f"Bearer {API_TOKEN}"}

# Test authenticated request
requests.get("https://api.github.com/users", headers=headers).json()

{'message': 'Bad credentials',
 'documentation_url': 'https://docs.github.com/rest',
 'status': '401'}

## 5. Pagination
GitHub returns max 30 events per page and provides a `Link` header with the `next` page URL.

In [5]:
response = requests.get(REPO_URL)

print(response.links)                    # Parsed Link header
print(response.links['next']['url'])     # Next page URL

{'next': {'url': 'https://api.github.com/repositories/419661684/events?page=2', 'rel': 'next'}, 'last': {'url': 'https://api.github.com/repositories/419661684/events?page=10', 'rel': 'last'}}
https://api.github.com/repositories/419661684/events?page=2


In [6]:
# Iterate through all pages
url = REPO_URL
while True:
    response = requests.get(url)
    data = response.json()
    print(f"{len(data)} events")

    if 'next' not in response.links:
        break
    url = response.links['next']['url']

30 events
30 events
30 events
30 events
30 events
30 events
30 events
30 events
30 events
30 events


## 6. Generator — Stream Pages Without Loading All Into Memory

In [7]:
def events_getter(url=REPO_URL, headers=None):
    """Yields one page of GitHub events at a time."""
    while True:
        response = requests.get(url, headers=headers or {})
        response.raise_for_status()
        yield response.json()

        if 'next' not in response.links:
            break
        url = response.links['next']['url']

In [8]:
# Test: count all fetched events
total = sum(len(page) for page in events_getter())
print(f"Total events: {total}")

Total events: 300


## 7. Load — PostgreSQL via `dlt`
> Configure your DB credentials in `~/.dlt/secrets.toml` before running.

In [ ]:
import dlt

pipeline = dlt.pipeline(
    pipeline_name="github_events",
    destination="postgres",
    dataset_name="github_data"
)

load_info = pipeline.run(
    events_getter(headers=headers),
    table_name="events",
    write_disposition="append"
)

print(load_info)